# E-Commerce Sales - Exploratory Data Analysis (EDA)

This notebook performs a comprehensive exploratory data analysis on the E-Commerce Sales dataset.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

## 2. Load the Dataset

In [ ]:
# Load the dataset
file_path = 'Data set.xlsx'
df = pd.read_excel(file_path)

print(f"Dataset loaded successfully!")
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

## 3. Data Inspection

In [ ]:
# Display dataset information
print("Dataset Information:")
print("="*50)
df.info()

print("\nColumn Names:")
print("="*50)
print(df.columns.tolist())

print("\nData Types:")
print("="*50)
print(df.dtypes)

print("\nDataset Shape:")
print("="*50)
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

## 4. Missing Values Check

In [ ]:
# Check for missing values
print("Missing Values Analysis:")
print("="*50)
missing_values = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing_values.index,
    'Missing Count': missing_values.values,
    'Missing Percentage': missing_percent.values
})

missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if len(missing_df) == 0:
    print("No missing values found!")
else:
    print(missing_df)
    
    # Visualize missing values
    fig, ax = plt.subplots(figsize=(10, 5))
    missing_df.plot(x='Column', y='Missing Percentage', kind='bar', ax=ax, legend=False)
    plt.title('Missing Values Percentage by Column', fontsize=14, fontweight='bold')
    plt.xlabel('Column Name')
    plt.ylabel('Missing Percentage (%)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 5. Duplicate Rows Check

In [ ]:
# Check for duplicates
print("Duplicate Rows Analysis:")
print("="*50)
duplicate_rows = df.duplicated().sum()
duplicate_all = df.duplicated(keep=False).sum()

print(f"Total duplicate rows: {duplicate_rows}")
print(f"Rows involved in duplication (keep=False): {duplicate_all}")

if duplicate_rows > 0:
    print(f"\nDuplicate rows found. Showing first 10:")
    print(df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10))
    
    # Remove duplicates
    df_cleaned = df.drop_duplicates()
    print(f"\nRows before removing duplicates: {len(df)}")
    print(f"Rows after removing duplicates: {len(df_cleaned)}")
    df = df_cleaned
else:
    print("No duplicate rows found!")

## 6. Data Cleaning

In [ ]:
# Data cleaning
print("Data Cleaning Process:")
print("="*50)

# Handle missing values if any
for col in df.columns:
    if df[col].dtype == 'object':
        if df[col].isnull().sum() > 0:
            df[col].fillna('Unknown', inplace=True)
    else:
        if df[col].isnull().sum() > 0:
            df[col].fillna(df[col].median(), inplace=True)

# Convert columns to appropriate types
for col in df.columns:
    if 'date' in col.lower():
        try:
            df[col] = pd.to_datetime(df[col])
        except:
            pass

print("Data cleaning completed!")
print(f"\nCurrent dataset shape: {df.shape}")
print(f"\nDataset head after cleaning:")
df.head()

## 7. Descriptive Statistics

In [ ]:
# Descriptive statistics for numerical columns
print("Descriptive Statistics (Numerical Columns):")
print("="*50)
numerical_stats = df.describe()
print(numerical_stats)

print("\n\nDescriptive Statistics (All Columns):")
print("="*50)
print(df.describe(include='all'))

## 8. Categorical Data Overview

In [ ]:
# Categorical columns analysis
print("Categorical Columns Analysis:")
print("="*50)

categorical_cols = df.select_dtypes(include='object').columns.tolist()

for col in categorical_cols:
    print(f"\n{col}:")
    print(f"Unique values: {df[col].nunique()}")
    print(f"Value counts:")
    print(df[col].value_counts())
    print("-" * 40)

## 9. Revenue Analysis

In [ ]:
# Identify revenue column (try common names)
revenue_col = None
for col in df.columns:
    if 'revenue' in col.lower() or 'total' in col.lower() or 'amount' in col.lower() or 'price' in col.lower() or 'sales' in col.lower():
        if df[col].dtype in ['float64', 'int64']:
            revenue_col = col
            break

if revenue_col:
    print(f"Revenue Column: {revenue_col}")
    print("="*50)
    print(f"Total Revenue: ${df[revenue_col].sum():,.2f}")
    print(f"Average Revenue: ${df[revenue_col].mean():,.2f}")
    print(f"Median Revenue: ${df[revenue_col].median():,.2f}")
    print(f"Min Revenue: ${df[revenue_col].min():,.2f}")
    print(f"Max Revenue: ${df[revenue_col].max():,.2f}")
    print(f"Standard Deviation: ${df[revenue_col].std():,.2f}")
    
    # Revenue distribution visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(df[revenue_col], bins=30, color='skyblue', edgecolor='black')
    axes[0].set_title(f'{revenue_col} Distribution', fontsize=12, fontweight='bold')
    axes[0].set_xlabel(revenue_col)
    axes[0].set_ylabel('Frequency')
    axes[0].grid(axis='y', alpha=0.3)
    
    axes[1].boxplot(df[revenue_col])
    axes[1].set_title(f'{revenue_col} Box Plot', fontsize=12, fontweight='bold')
    axes[1].set_ylabel(revenue_col)
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Revenue column not found. Numerical columns available:")
    print(df.select_dtypes(include=['float64', 'int64']).columns.tolist())

## 10. Monthly Revenue Trend

In [ ]:
# Monthly revenue trend
date_col = None
for col in df.columns:
    if 'date' in col.lower() or 'time' in col.lower():
        if df[col].dtype == 'datetime64[ns]':
            date_col = col
            break

if date_col and revenue_col:
    print(f"Monthly Revenue Trend:")
    print("="*50)
    
    df['Year_Month'] = df[date_col].dt.to_period('M')
    monthly_revenue = df.groupby('Year_Month')[revenue_col].sum().sort_index()
    
    print(monthly_revenue)
    
    # Plot monthly trend
    fig, ax = plt.subplots(figsize=(14, 6))
    monthly_revenue.plot(ax=ax, marker='o', linewidth=2, markersize=8, color='green')
    ax.set_title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
    ax.set_xlabel('Month')
    ax.set_ylabel('Revenue ($)')
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    print(f"\nAverage Monthly Revenue: ${monthly_revenue.mean():,.2f}")
    print(f"Peak Month Revenue: ${monthly_revenue.max():,.2f}")
    print(f"Lowest Month Revenue: ${monthly_revenue.min():,.2f}")
else:
    print("Date column or Revenue column not found for trend analysis.")

## 11. Product-wise Revenue Analysis

In [ ]:
# Product-wise revenue
product_col = None
for col in df.columns:
    if 'product' in col.lower():
        product_col = col
        break

if product_col and revenue_col:
    print(f"Product-wise Revenue Analysis:")
    print("="*50)
    
    product_revenue = df.groupby(product_col)[revenue_col].agg(['sum', 'count', 'mean']).round(2)
    product_revenue.columns = ['Total Revenue', 'Transaction Count', 'Avg Revenue']
    product_revenue = product_revenue.sort_values('Total Revenue', ascending=False)
    
    print(product_revenue)
    
    # Visualize top products
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    top_products = product_revenue.head(10)
    axes[0].barh(range(len(top_products)), top_products['Total Revenue'], color='coral')
    axes[0].set_yticks(range(len(top_products)))
    axes[0].set_yticklabels(top_products.index)
    axes[0].set_xlabel('Total Revenue ($)')
    axes[0].set_title('Top 10 Products by Revenue', fontsize=12, fontweight='bold')
    axes[0].invert_yaxis()
    
    axes[1].barh(range(len(top_products)), top_products['Transaction Count'], color='skyblue')
    axes[1].set_yticks(range(len(top_products)))
    axes[1].set_yticklabels(top_products.index)
    axes[1].set_xlabel('Transaction Count')
    axes[1].set_title('Top 10 Products by Transaction Count', fontsize=12, fontweight='bold')
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.show()
else:
    print("Product column or Revenue column not found.")

## 12. Customer Analysis

In [ ]:
# Customer analysis
customer_col = None
for col in df.columns:
    if 'customer' in col.lower() or 'user' in col.lower() or 'buyer' in col.lower():
        customer_col = col
        break

if customer_col and revenue_col:
    print(f"Customer Analysis:")
    print("="*50)
    
    total_customers = df[customer_col].nunique()
    print(f"Total Unique Customers: {total_customers}")
    
    customer_revenue = df.groupby(customer_col)[revenue_col].agg(['sum', 'count', 'mean']).round(2)
    customer_revenue.columns = ['Total Spent', 'Purchase Count', 'Avg Purchase']
    customer_revenue = customer_revenue.sort_values('Total Spent', ascending=False)
    
    print(f"\nTop 10 Customers by Revenue:")
    print(customer_revenue.head(10))
    
    print(f"\nCustomer Spending Statistics:")
    print(f"Average Customer Spend: ${customer_revenue['Total Spent'].mean():,.2f}")
    print(f"Median Customer Spend: ${customer_revenue['Total Spent'].median():,.2f}")
    print(f"Max Customer Spend: ${customer_revenue['Total Spent'].max():,.2f}")
    
    # Visualize customer distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(customer_revenue['Total Spent'], bins=30, color='purple', edgecolor='black', alpha=0.7)
    axes[0].set_title('Customer Spending Distribution', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Total Spent ($)')
    axes[0].set_ylabel('Frequency')
    axes[0].grid(axis='y', alpha=0.3)
    
    axes[1].scatter(customer_revenue['Purchase Count'], customer_revenue['Total Spent'], alpha=0.6, color='darkblue')
    axes[1].set_title('Customer Purchase Frequency vs Total Spending', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Purchase Count')
    axes[1].set_ylabel('Total Spent ($)')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Customer column or Revenue column not found.")

## 13. Payment Method Analysis

In [ ]:
# Payment method analysis
payment_col = None
for col in df.columns:
    if 'payment' in col.lower() or 'method' in col.lower():
        payment_col = col
        break

if payment_col and revenue_col:
    print(f"Payment Method Analysis:")
    print("="*50)
    
    payment_revenue = df.groupby(payment_col)[revenue_col].agg(['sum', 'count', 'mean']).round(2)
    payment_revenue.columns = ['Total Revenue', 'Transaction Count', 'Avg Transaction']
    payment_revenue = payment_revenue.sort_values('Total Revenue', ascending=False)
    
    print(payment_revenue)
    
    # Visualize payment methods
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    colors = plt.cm.Set3(range(len(payment_revenue)))
    
    axes[0].pie(payment_revenue['Total Revenue'], labels=payment_revenue.index, autopct='%1.1f%%', colors=colors)
    axes[0].set_title('Revenue Distribution by Payment Method', fontsize=12, fontweight='bold')
    
    axes[1].pie(payment_revenue['Transaction Count'], labels=payment_revenue.index, autopct='%1.1f%%', colors=colors)
    axes[1].set_title('Transaction Count by Payment Method', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
else:
    print("Payment method column or Revenue column not found.")

## 14. Referral Source Analysis

In [ ]:
# Referral source analysis
referral_col = None
for col in df.columns:
    if 'referral' in col.lower() or 'source' in col.lower() or 'channel' in col.lower():
        referral_col = col
        break

if referral_col and revenue_col:
    print(f"Referral Source Analysis:")
    print("="*50)
    
    referral_revenue = df.groupby(referral_col)[revenue_col].agg(['sum', 'count', 'mean']).round(2)
    referral_revenue.columns = ['Total Revenue', 'Transaction Count', 'Avg Transaction']
    referral_revenue = referral_revenue.sort_values('Total Revenue', ascending=False)
    
    print(referral_revenue)
    
    # Visualize referral sources
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    axes[0].bar(range(len(referral_revenue)), referral_revenue['Total Revenue'], color='lightcoral')
    axes[0].set_xticks(range(len(referral_revenue)))
    axes[0].set_xticklabels(referral_revenue.index, rotation=45, ha='right')
    axes[0].set_ylabel('Total Revenue ($)')
    axes[0].set_title('Revenue by Referral Source', fontsize=12, fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)
    
    axes[1].bar(range(len(referral_revenue)), referral_revenue['Transaction Count'], color='lightyellow', edgecolor='black')
    axes[1].set_xticks(range(len(referral_revenue)))
    axes[1].set_xticklabels(referral_revenue.index, rotation=45, ha='right')
    axes[1].set_ylabel('Transaction Count')
    axes[1].set_title('Transaction Count by Referral Source', fontsize=12, fontweight='bold')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Referral source column or Revenue column not found.")

## 15. Coupon Analysis

In [ ]:
# Coupon analysis
coupon_col = None
for col in df.columns:
    if 'coupon' in col.lower() or 'discount' in col.lower() or 'promo' in col.lower():
        coupon_col = col
        break

if coupon_col:
    print(f"Coupon/Discount Analysis:")
    print("="*50)
    
    if df[coupon_col].dtype in ['float64', 'int64']:
        print(f"Total Coupon/Discount Value: ${df[coupon_col].sum():,.2f}")
        print(f"Average Coupon/Discount: ${df[coupon_col].mean():,.2f}")
        print(f"Max Coupon/Discount: ${df[coupon_col].max():,.2f}")
        print(f"Min Coupon/Discount: ${df[coupon_col].min():,.2f}")
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        axes[0].hist(df[coupon_col], bins=30, color='gold', edgecolor='black')
        axes[0].set_title('Coupon/Discount Distribution', fontsize=12, fontweight='bold')
        axes[0].set_xlabel('Coupon/Discount Amount ($)')
        axes[0].set_ylabel('Frequency')
        axes[0].grid(axis='y', alpha=0.3)
        
        if revenue_col:
            axes[1].scatter(df[coupon_col], df[revenue_col], alpha=0.5, color='darkgreen')
            axes[1].set_title('Coupon/Discount vs Revenue', fontsize=12, fontweight='bold')
            axes[1].set_xlabel('Coupon/Discount ($)')
            axes[1].set_ylabel('Revenue ($)')
            axes[1].grid(alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    else:
        print(f"Unique Coupons/Discounts: {df[coupon_col].nunique()}")
        coupon_usage = df[coupon_col].value_counts()
        print(f"\nTop 10 Coupons/Discounts:")
        print(coupon_usage.head(10))
else:
    print("Coupon/Discount column not found.")

## 16. Correlation Heatmap

In [ ]:
# Correlation analysis
print("Correlation Analysis:")
print("="*50)

numerical_df = df.select_dtypes(include=[np.number])

if len(numerical_df.columns) > 1:
    correlation_matrix = numerical_df.corr()
    
    print("Correlation Matrix:")
    print(correlation_matrix)
    
    # Visualize correlation heatmap
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', 
                square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax)
    plt.title('Correlation Heatmap of Numerical Features', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Find strong correlations
    print("\nStrong Correlations (|r| > 0.7):")
    strong_corr = []
    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            if abs(correlation_matrix.iloc[i, j]) > 0.7:
                strong_corr.append((correlation_matrix.columns[i], 
                                   correlation_matrix.columns[j], 
                                   correlation_matrix.iloc[i, j]))
    
    if strong_corr:
        for var1, var2, corr in strong_corr:
            print(f"{var1} <-> {var2}: {corr:.3f}")
    else:
        print("No strong correlations found.")
else:
    print("Not enough numerical columns for correlation analysis.")

## 17. Outlier Detection (IQR Method)

In [ ]:
# Outlier detection using IQR method
print("Outlier Detection (IQR Method):")
print("="*50)

numerical_df = df.select_dtypes(include=[np.number])
outliers_detected = {}

fig, axes = plt.subplots(int(np.ceil(len(numerical_df.columns) / 2)), 2, figsize=(14, 4 * int(np.ceil(len(numerical_df.columns) / 2))))
axes = axes.flatten()

for idx, col in enumerate(numerical_df.columns):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outliers_detected[col] = len(outliers)
    
    print(f"\n{col}:")
    print(f"  Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
    print(f"  Lower Bound: {lower_bound:.2f}, Upper Bound: {upper_bound:.2f}")
    print(f"  Outliers Detected: {len(outliers)} ({(len(outliers)/len(df)*100):.2f}%)")
    
    # Box plot
    axes[idx].boxplot(df[col])
    axes[idx].set_title(f'{col} - Box Plot', fontweight='bold')
    axes[idx].set_ylabel(col)
    axes[idx].grid(axis='y', alpha=0.3)

# Hide unused subplots
for idx in range(len(numerical_df.columns), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

## 18. Additional Visualizations

In [ ]:
# Distribution of key numerical columns
print("Distribution Analysis of Key Numerical Columns:")
print("="*50)

numerical_df = df.select_dtypes(include=[np.number])
n_cols = min(len(numerical_df.columns), 6)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, col in enumerate(numerical_df.columns[:n_cols]):
    axes[idx].hist(df[col], bins=25, color='steelblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{col} Distribution', fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(axis='y', alpha=0.3)

# Hide unused subplots
for idx in range(n_cols, 6):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

## 19. Business Insights and Recommendations

In [ ]:
print("\n" + "="*70)
print("BUSINESS INSIGHTS AND RECOMMENDATIONS")
print("="*70)

insights = []

# Insight 1: Dataset Overview
insights.append(f"\n1. DATASET OVERVIEW:")
insights.append(f"   - Total Records: {len(df):,}")
insights.append(f"   - Total Columns: {len(df.columns)}")
insights.append(f"   - Date Range: {df[date_col].min() if date_col else 'N/A'} to {df[date_col].max() if date_col else 'N/A'}")

# Insight 2: Revenue Performance
if revenue_col:
    insights.append(f"\n2. REVENUE PERFORMANCE:")
    insights.append(f"   - Total Revenue: ${df[revenue_col].sum():,.2f}")
    insights.append(f"   - Average Transaction: ${df[revenue_col].mean():,.2f}")
    insights.append(f"   - Revenue Volatility (Std Dev): ${df[revenue_col].std():,.2f}")

# Insight 3: Customer Insights
if customer_col:
    insights.append(f"\n3. CUSTOMER INSIGHTS:")
    insights.append(f"   - Total Unique Customers: {df[customer_col].nunique():,}")
    if revenue_col:
        avg_customer_value = df.groupby(customer_col)[revenue_col].sum().mean()
        insights.append(f"   - Average Customer Lifetime Value: ${avg_customer_value:,.2f}")

# Insight 4: Product Performance
if product_col and revenue_col:
    insights.append(f"\n4. PRODUCT PERFORMANCE:")
    top_product = product_revenue.index[0]
    top_revenue = product_revenue.iloc[0]['Total Revenue']
    insights.append(f"   - Best Performing Product: {top_product}")
    insights.append(f"   - Top Product Revenue: ${top_revenue:,.2f}")
    insights.append(f"   - Total Unique Products: {df[product_col].nunique()}")

# Insight 5: Payment Methods
if payment_col:
    insights.append(f"\n5. PAYMENT METHOD INSIGHTS:")
    most_used_payment = payment_revenue.index[0]
    insights.append(f"   - Most Used Payment Method: {most_used_payment}")
    insights.append(f"   - Payment Methods Available: {df[payment_col].nunique()}")

# Insight 6: Data Quality
insights.append(f"\n6. DATA QUALITY:")
insights.append(f"   - Missing Values: {df.isnull().sum().sum()}")
insights.append(f"   - Duplicate Records: {df.duplicated().sum()}")
insights.append(f"   - Data Completeness: {((1 - df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100):.2f}%")

# Print all insights
for insight in insights:
    print(insight)

print("\n" + "="*70)
print("RECOMMENDATIONS:")
print("="*70)

recommendations = [
    "1. Focus on high-revenue products and customers for retention and upselling.",
    "2. Analyze seasonal trends to optimize inventory and marketing campaigns.",
    "3. Investigate payment method preferences to optimize checkout experience.",
    "4. Identify and address outliers in revenue to understand exceptional transactions.",
    "5. Develop targeted promotions based on referral source performance.",
    "6. Monitor coupon effectiveness and ROI to optimize discount strategies.",
    "7. Implement customer segmentation for personalized marketing campaigns.",
    "8. Continuously monitor data quality and implement automated validation checks."
]

for rec in recommendations:
    print(rec)

print("\n" + "="*70)

## 20. Summary

In [ ]:
print("\n" + "="*70)
print("EDA ANALYSIS SUMMARY")
print("="*70)
print("\nThis exploratory data analysis has covered:")
print("✓ Data Loading and Inspection")
print("✓ Missing Values and Duplicates Detection")
print("✓ Data Cleaning and Preprocessing")
print("✓ Descriptive Statistics")
print("✓ Revenue Analysis and Trends")
print("✓ Product Performance Analysis")
print("✓ Customer Behavior Analysis")
print("✓ Payment Method Distribution")
print("✓ Referral Source Performance")
print("✓ Coupon and Discount Analysis")
print("✓ Correlation Analysis")
print("✓ Outlier Detection")
print("✓ Comprehensive Visualizations")
print("✓ Business Insights and Recommendations")
print("\n" + "="*70)
print("Analysis complete! All visualizations and insights have been generated.")
print("="*70)